# this 与调用方式

学习目标：能从调用形式解释 this，修复方法提取和回调丢失接收者的问题，并识别构造调用。

前置知识：普通函数、箭头函数、对象方法、闭包和严格模式。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 文件使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/12-this-and-calls/。

1. [call-sites.mjs](scripts/12-this-and-calls/call-sites.mjs)：同一函数在不同调用位置的 this。
2. [non-strict.cjs](scripts/12-this-and-calls/non-strict.cjs)：非严格函数默认接收者与装箱。
3. [lexical-this.mjs](scripts/12-this-and-calls/lexical-this.mjs)：方法内部箭头函数与对象中的箭头属性。
4. [explicit-binding.mjs](scripts/12-this-and-calls/explicit-binding.mjs)：显式绑定、部分参数与回调。
5. [detached-error.mjs](scripts/12-this-and-calls/detached-error.mjs)：方法提取后 this 为 undefined 的独立反例。
6. [construct.mjs](scripts/12-this-and-calls/construct.mjs)：构造目标与绑定函数的 new 调用。

## 1 普通调用和方法调用

不要沿函数的定义位置猜 this，先在调用处找有没有接收者。下图对照同一 receiver 函数的三种调用。

this 是本次调用的接收者。对本节的普通严格函数，first.receiver() 保留 first 这个接收者；先取出函数再写 detached()，就没有这层调用关系，this 为 undefined。定义函数的文件或最初存放函数的对象不决定这次调用的 this。点号与方括号属性调用遵循相同原则。

模块中的函数处于严格模式。严格普通函数保留调用给定的 this，不把 null、undefined 替换成全局对象，也不把原始值装箱。模块顶层 this 是 undefined；这与 globalThis 仍然可用是两回事。

![相同函数，接收者由调用形式决定。以下三行都是 ES 模块中的普通严格函数调用。](image/illustration/12-01-this-call-site.svg)

图示说明：图限于普通严格函数；箭头函数、bind 和 new 的规则在各自小节单独讨论。

对应下面 first、second 和 detached 的调用，核对对象是否相同；call(null) 与 call(7) 再观察严格函数保留显式接收者。

[call-sites.mjs](scripts/12-this-and-calls/call-sites.mjs)：

```javascript
function receiver() { return this; }
const first = { receiver };
const second = { receiver: first.receiver };
console.log(first.receiver() === first, second.receiver() === second);
const detached = first.receiver;
console.log(detached() === undefined, this === undefined);
console.log(receiver.call(null) === null, receiver.call(7) === 7);

// 按本例输入运行，输出依次为：
// true true
// true true
// true true
```

Step 1：运行本节示例。

```bash
node scripts/12-this-and-calls/call-sites.mjs
```

## 2 非严格普通函数的转换

为对照而保留的 .cjs 文件没有 "use strict" 指令，其中普通函数在非严格模式运行。它收到 undefined 或 null 时使用该函数所属环境的全局 this 值；收到数字等原始值时使用包装对象。是否转换取决于被调用函数自身的模式，而不是调用方是否严格。

CommonJS 的文件顶层由 Node.js 包装，不能把这里的非严格函数规则直接当成 CommonJS 顶层 this 规则，更不能套到浏览器模块。新示例继续优先使用 .mjs 明确上下文。

[non-strict.cjs](scripts/12-this-and-calls/non-strict.cjs)：

```javascript
function receiver() { return this; }
console.log(receiver() === globalThis, receiver.call(null) === globalThis);
const boxed = receiver.call(7);
console.log(typeof boxed, boxed.valueOf());
function strictReceiver() { "use strict"; return this; }
console.log(strictReceiver() === undefined, strictReceiver.call(7) === 7);

// 按本例输入运行，输出依次为：
// true true
// object 7
// true true
```

Step 1：运行本节示例。

```bash
node scripts/12-this-and-calls/non-strict.cjs
```

## 3 箭头函数捕获外层 this

箭头函数没有自己的 this 绑定，读取定义时所在词法环境的 this。把箭头函数放进某个对象，不会让这个对象成为它的 this。适合的用法是在普通方法内部创建箭头回调，让回调继续使用这次方法调用的接收者。

call、apply、bind 无法替换箭头函数捕获的 this。模块顶层创建的箭头函数捕获模块顶层 this；需要通过调用对象选择接收者的操作应使用普通方法。

[lexical-this.mjs](scripts/12-this-and-calls/lexical-this.mjs)：

```javascript
const topArrow = () => this;
const misleading = { topArrow };
console.log(misleading.topArrow() === undefined);
const session = {
  prefix: "JS",
  labelAll(names) {
    return names.map(name => this.prefix + ":" + name);
  },
  makeReader() { return () => this.prefix; }
};
console.log(session.labelAll(["集合", "模块"]).join(","));
const read = session.makeReader();
console.log(read.call({ prefix: "TS" }));

// 按本例输入运行，输出依次为：
// true
// JS:集合,JS:模块
// JS
```

Step 1：运行本节示例。

```bash
node scripts/12-this-and-calls/lexical-this.mjs
```

## 4 call、apply、bind 与回调修复

call 立即调用函数，其后的参数逐个传入；apply 立即调用，参数由数组或类数组对象提供；bind 返回新函数，保存 this 和预先提供的参数，此时尚未执行函数体。再次用 call 调用绑定函数不会改写原先绑定的 this。

回调如何调用由调用者决定。下面 run 采用 callback()，因此提取方法后会丢失接收者。可选择 bind 固定接收者，或传入箭头包装，在包装中明确执行对象方法。bind 还固定原函数；箭头包装中的属性查找发生在包装被调用时，两者在后来替换方法时存在差异。

[explicit-binding.mjs](scripts/12-this-and-calls/explicit-binding.mjs)：

```javascript
function describe(prefix, suffix) { return prefix + this.title + suffix; }
const item = { title: "集合" };
console.log(describe.call(item, "[", "]"));
console.log(describe.apply(item, ["<", ">"]));
const bound = describe.bind(item, "(");
console.log(bound(")"), bound.call({ title: "其他" }, ")"));
const run = callback => callback();
const counter = { value: 0, increment() { return ++this.value; } };
console.log(run(counter.increment.bind(counter)));
console.log(run(() => counter.increment()));

// 按本例输入运行，输出依次为：
// [集合]
// <集合>
// (集合) (集合)
// 1
// 2
```

Step 1：运行本节示例。

```bash
node scripts/12-this-and-calls/explicit-binding.mjs
```

[detached-error.mjs](scripts/12-this-and-calls/detached-error.mjs)：

```javascript
const counter = { value: 0, increment() { return ++this.value; } };
const callback = counter.increment;
callback();

// 独立运行：退出状态为 1；诊断包含 TypeError；Cannot read properties of undefined；value。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/12-this-and-calls/detached-error.mjs
```

## 5 new 调用与 new.target

对可构造的普通函数使用 new 时，函数体中的 this 通常是新创建的对象；函数默认返回该实例，显式返回另一个对象则会替代它。构造函数不应混用“初始化实例”和“返回其他对象”两种含义。

new.target 表示这次构造所使用的目标，普通调用时为 undefined，可以用于检测调用方式。new 调用绑定函数时，预设参数仍生效，但 bind 保存的 this 被忽略。箭头函数不可构造，bind 也不能把它变成构造函数。

[construct.mjs](scripts/12-this-and-calls/construct.mjs)：

```javascript
function Entry(title) {
  if (!new.target) return "需要 new";
  this.title = title;
  this.target = new.target.name;
}
const unrelated = {};
const BoundEntry = Entry.bind(unrelated, "迭代");
const entry = new BoundEntry();
console.log(Entry("普通调用"));
console.log(entry.title, entry.target, entry instanceof Entry);
console.log(Object.hasOwn(unrelated, "title"));
function Replacement() { this.discarded = true; return { selected: true }; }
console.log(new Replacement().selected);

// 按本例输入运行，输出依次为：
// 需要 new
// 迭代 Entry true
// false
// true
```

Step 1：运行本节示例。

```bash
node scripts/12-this-and-calls/construct.mjs
```

## 本章小结

- 判断 this 先看函数种类与调用位置，再看被调用函数的严格模式。
- 箭头函数沿词法环境取 this；普通函数可用 call、apply、bind 明确接收者。
- new 是构造调用，new.target 可区分普通调用与构造调用。

## 练习

1. 让同一个普通函数作为两个对象的方法，分别返回对象中的 title。可核对标准：两次调用得到不同标题，提取后普通调用不能自动恢复原对象。
2. 分别使用 bind 和箭头包装修复 detached-error.mjs。可核对标准：两份修复都正常退出，counter.value 增加一次。
3. 对 BoundEntry 再指定一个不同的 this 后用 new 调用。可核对标准：新实例仍由 Entry 初始化，两个指定对象都没有新增 title 属性。

## 参考与引用来源

- TC39 官方 ECMAScript 2025 分页版：[§10.2.1.2 严格、非严格及词法 this](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ordinarycallbindthis)；[§20.2.3 call/apply/bind](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-properties-of-the-function-prototype-object)；[§10.2.2 构造调用](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ecmascript-function-objects-construct-argumentslist-newtarget)；[§10.4.1 绑定函数调用与构造](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-bound-function-exotic-objects)；[§15.3 箭头函数](https://tc39.es/ecma262/2025/multipage/ecmascript-language-functions-and-classes.html#sec-arrow-function-definitions)。